# Walmart Sales

link: https://www.kaggle.com/datasets/mikhail1681/walmart-sales

In [1]:
import pandas as pd
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
import plotly.graph_objects as go

In [2]:
df = pd.read_csv(
    "../data/01_raw/Walmart_Sales.csv", parse_dates=["Date"], dayfirst=True
)
df

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,1,2010-02-05,1643690.90,0,42.31,2.572,211.096358,8.106
1,1,2010-02-12,1641957.44,1,38.51,2.548,211.242170,8.106
2,1,2010-02-19,1611968.17,0,39.93,2.514,211.289143,8.106
3,1,2010-02-26,1409727.59,0,46.63,2.561,211.319643,8.106
4,1,2010-03-05,1554806.68,0,46.50,2.625,211.350143,8.106
...,...,...,...,...,...,...,...,...
6430,45,2012-09-28,713173.95,0,64.88,3.997,192.013558,8.684
6431,45,2012-10-05,733455.07,0,64.89,3.985,192.170412,8.667
6432,45,2012-10-12,734464.36,0,54.47,4.000,192.327265,8.667
6433,45,2012-10-19,718125.53,0,56.47,3.969,192.330854,8.667


In [ ]:
PREDICTION_LENGTH=13
TARGET="Weekly_Sales"

In [ ]:
train_data = TimeSeriesDataFrame.from_data_frame(
    df, id_column="Store", timestamp_column="Date"
)

train_split, test_split = train_data.train_test_split(
    prediction_length=PREDICTION_LENGTH
)

predictor = TimeSeriesPredictor(
    prediction_length=PREDICTION_LENGTH,
    quantile_levels=[0.025, 0.05, 0.5, 0.95 ,0.975],
    target=TARGET,
    eval_metric="WAPE",
)

predictor.fit(
    train_split,
    hyperparameters={
        "RecursiveTabular": {
            "tabular_hyperparameters": {"GBM": {}, "XGB": {}, "CAT": {}}
        },
        "Naive": {},
        "SeasonalNaive": {},
        "PatchTST": {},
        "ETS": {},
        "ADIDA": {},
        "DLinear": {},
        "Toto": {},
        "ARIMA": {},
        "Theta": {},
        "DeepAR": {},
        "NPTS": {},
        "TemporalFusionTransformer": {},
        "Chronos2": {},
        "SimpleFeedForward": {},
        "TiDE":{},
        "WaveNet": {},
    },
)

Beginning AutoGluon training...
AutoGluon will save models to '/home/cezartdev/Documents/cezartdev/personal/walmart_sales_benchmark_mlops/notebooks/AutogluonModels/ag-20260518_005030'
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Thu Mar  5 00:10:35 UTC 2026
CPU Count:          12
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 11.62/11.62 GB
Total GPU Memory:   Free: 11.62 GB, Allocated: 0.00 GB, Total: 11.62 GB
GPU Count:          1
Memory Avail:       21.85 GB / 31.15 GB (70.2%)
Disk Space Avail:   111.03 GB / 230.30 GB (48.2%)

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WAPE,
 'hyperparameters': {'ADIDA': {},
                     'ARIMA': {},
                     'Chronos2': {},
                     'DLinear': {},
                     'DeepAR': {},
                     '

In [5]:
leaderboard = predictor.leaderboard(train_split)

Additional data provided, testing on additional data. Resulting leaderboard will be sorted according to test score (`score_test`).


In [6]:
leaderboard

,model,score_test,score_val,pred_time_test,pred_time_val,fit_time_marginal,fit_order
0,WeightedEnsemble,-0.029279,-0.029319,1.211972,0.529034,0.228055,18
1,TiDE,-0.032681,-0.032681,0.039852,0.030486,87.305403,11
2,Chronos2,-0.034291,-0.034291,1.075165,0.411704,2.478173,7
3,PatchTST,-0.034841,-0.034841,0.026372,0.019022,36.763068,10
4,RecursiveTabular,-0.044680,-0.044680,0.045911,0.041179,0.308654,3
5,TemporalFusionTransformer,-0.049835,-0.049835,0.061735,0.023115,20.605269,8
6,DeepAR,-0.051110,-0.051562,0.069346,0.066399,28.794412,9
7,WaveNet,-0.051420,-0.052694,0.053103,0.080233,15.436402,12
8,SimpleFeedForward,-0.052638,-0.052638,0.018024,0.016023,19.479304,16
9,ARIMA,-0.052734,-0.052734,0.085852,0.085531,0.006951,15


In [7]:
predictions = predictor.predict(train_split, model="WeightedEnsemble")

In [10]:
store_id = 2

real_data = test_split.loc[store_id]
predictions_store = predictions.loc[store_id]

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=real_data.index,
        y=real_data[TARGET],
        name="Actual Sales (Full)",
        mode="lines",
        line=dict(color="#1f77b4", width=2),
    )
)

fig.add_trace(
    go.Scatter(
        x=predictions_store.index,
        y=predictions_store["mean"],
        name="Forecast (Mean)",
        mode="lines",
        line=dict(color="#ff7f0e", width=2, dash="dash"),
    )
)

fig.add_trace(
    go.Scatter(
        x=predictions_store.index,
        y=predictions_store["0.975"],
        name="Upper Bound (p97.5)",
        mode="lines",
        line=dict(width=0),
        showlegend=False,
    )
)

fig.add_trace(
    go.Scatter(
        x=predictions_store.index,
        y=predictions_store["0.025"],
        name="95% Confidence Interval (p2.5 - p97.5)",
        mode="lines",
        fill="tonexty",
        fillcolor="rgba(255, 127, 14, 0.15)",
        line=dict(width=0),
    )
)

fig.add_trace(
    go.Scatter(
        x=predictions_store.index,
        y=predictions_store["0.95"],
        name="Upper Bound (p95)",
        mode="lines",
        line=dict(width=0),
        showlegend=False,
    )
)

fig.add_trace(
    go.Scatter(
        x=predictions_store.index,
        y=predictions_store["0.05"],
        name="90% Confidence Interval (p5 - p95)",
        mode="lines",
        fill="tonexty",
        fillcolor="rgba(200, 100, 14, 0.15)",
        line=dict(width=0),
    )
)

fig.update_layout(
    title=f"Backtesting (Actual vs Forecast) - Store {store_id}",
    xaxis_title="Date",
    yaxis_title="Sales ($)",
    template="plotly_white",
    hovermode="x unified",
)
fig.show()